In [22]:
print(cfg["data"]["dir"])
print(cfg["data"]["topology"])
print(cfg["data"]["kind"])
print(cfg["data"]["stride"])

/Users/chloe/Desktop/project/msm/msm_agent/msm_agent/ala2_test_data
/Users/chloe/Desktop/project/msm/msm_agent/msm_agent/ala2_test_data/ala2.pdb
xtc
1


In [12]:
import mdtraj as md
traj = md.load("/Users/chloe/Desktop/project/msm/msm_agent/msm_agent/ala2_test_data/ala2-1ps-00.xtc",
               top="/Users/chloe/Desktop/project/msm/msm_agent/msm_agent/ala2_test_data/ala2.pdb",
               stride=1)
traj.n_frames

10001

In [13]:
import yaml
from pathlib import Path
from msm_agent.stages import stage_features, stage_tica_sweep, stage_tica_fit

cfg = yaml.safe_load(open("/Users/chloe/Desktop/project/msm/msm_agent/examples/interactive_pipeline.yaml"))
run_dir = Path(cfg["run"]["output_dir"]) / "debug_run_tmp"   # or create a timestamped run dir yourself
run_dir.mkdir(parents=True, exist_ok=True)


In [14]:
features, dt_ns, feat_meta = stage_features(cfg, run_dir, force=True)
feat_meta

{'n_trajs': 2,
 'feat_dim': 8,
 'traj_len_frames_min': 1,
 'traj_len_frames_max': 1,
 'total_frames': 2,
 'dim_consistent': True,
 'any_nan': False,
 'any_inf': False,
 'kind': 'xtc',
 'n_files': 2,
 'stride': 1,
 'dt_ns_effective': 0.001,
 'features_dir': '/Users/chloe/Desktop/project/msm/msm_agent/notebook/runs/debug_run_tmp/features',
 'run_dir': '/Users/chloe/Desktop/project/msm/msm_agent/notebook/runs/debug_run_tmp'}

In [15]:
import numpy as np
from msm_agent.metrics import (
    ck_test,
    compute_occupancy_stats,
    compute_tica_its,
    compute_transition_sparsity,
    compute_msm_its,
    its_plateau_metric,
    grade_run,
    suggest_fixes,
)
from msm_agent.plots import (
    plot_tica_density_hexbin,
    plot_free_energy,
    plot_occupancy_hist,
    plot_its_curve,
    plot_macro_overlay,
)

# --- load data and featurize
#features, dt_ps_effective = _load_feature(cfg, run_dir)
#dt_ns = dt_ps_effective / 1000.0  # ps -> ns
n_trajs = len(features)
traj_lens = [len(x) for x in features]
print("n_trajs:", n_trajs)
print("traj_lens:", traj_lens)

# check features

# --- tICA
tica_cfg = cfg["tica"]
tica_lag_list = np.linspace(int(tica_cfg["lag_time_frames_range"][0]),int(tica_cfg["lag_time_frames_range"][1]),num=20, dtype=int)
tica_its = compute_tica_its(features, tica_lag_list, n_components=int(tica_cfg["n_components"]), dt_ns=dt_ns)
# dict[lagtime, its_list]; lagtime in ns, its_list in ns
plot_its_curve(
    tica_its,
    outpath=run_dir / "figs" / "tica_its_curve.png",
    top_k=int(cfg["gates"]["plateau_k"]),
)

n_trajs: 2
traj_lens: [1, 1]


ValueError: All sequences were shorter than the lag time, 1

In [8]:
tica_sweep = stage_tica_sweep(cfg, run_dir, features, dt_ns=dt_ns)
tica_sweep["tica_its_plot"]

ValueError: All sequences were shorter than the lag time, 1

In [4]:
import glob, os, mdtraj as md

data_dir = cfg["data"]["dir"]
kind = cfg["data"]["kind"]
top = cfg["data"]["topology"]

files = sorted(glob.glob(os.path.join(data_dir, f"*.{kind}")))
print("FILES:", files)

for f in files:
    t = md.load(f, top=top, stride=1)
    print("RAW frames:", os.path.basename(f), t.n_frames)

FILES: ['/Users/chloe/Desktop/project/msm/msm_agent/msm_agent/ala2_test_data/ala2-1ps-00.xtc', '/Users/chloe/Desktop/project/msm/msm_agent/msm_agent/ala2_test_data/ala2-1ps-01.xtc']
RAW frames: ala2-1ps-00.xtc 10001
RAW frames: ala2-1ps-01.xtc 10001


In [5]:
from msm_agent.stages import stage_features

features, dt_ns, meta = stage_features(cfg, run_dir, force=True)
print(meta)
print("FEATURE shapes:", [x.shape for x in features])

{'n_trajs': 2, 'feat_dim': 8, 'traj_len_frames_min': 1, 'traj_len_frames_max': 1, 'total_frames': 2, 'dim_consistent': True, 'any_nan': False, 'any_inf': False, 'kind': 'xtc', 'n_files': 2, 'stride': 1, 'dt_ns_effective': 0.001, 'features_dir': '/Users/chloe/Desktop/project/msm/msm_agent/notebook/runs/debug_run_tmp/features', 'run_dir': '/Users/chloe/Desktop/project/msm/msm_agent/notebook/runs/debug_run_tmp'}
FEATURE shapes: [(1, 8), (1, 8)]


In [11]:
tics, tica_meta = stage_tica_fit(cfg, run_dir, features, tica_lag_frames=tica_sweep["tica_lag_list_frames"][0])
tica_meta

NameError: name 'tica_sweep' is not defined